In [1]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report, accuracy_score
import numpy as np

In [3]:
df = pd.read_csv("../insurance.csv")

In [4]:
df.sample(5)

,age,weight,height,income_lpa,smoker,city,occupation,insurance_premium_category
9,58,74.4,1.73,43.07,False,Pune,business_owner,Low
24,50,54.2,1.66,18.60,False,Mysore,private_job,Medium
81,41,82.6,1.61,22.19,True,Mysore,freelancer,High
40,44,57.0,1.53,40.19,True,Pune,unemployed,Medium
48,36,94.8,1.66,32.69,True,Chennai,unemployed,Medium


In [5]:
df['occupation'].unique()

array(['retired', 'freelancer', 'student', 'government_job',
       'business_owner', 'unemployed', 'private_job'], dtype=object)

In [6]:
df['city'].unique()

array(['Jaipur', 'Chennai', 'Indore', 'Mumbai', 'Kota', 'Hyderabad',
       'Delhi', 'Chandigarh', 'Pune', 'Kolkata', 'Lucknow', 'Gaya',
       'Jalandhar', 'Mysore', 'Bangalore'], dtype=object)

In [8]:
df['insurance_premium_category'].unique()

array(['High', 'Low', 'Medium'], dtype=object)

In [9]:
df_feat = df.copy()

In [10]:
df_feat["bmi"] = df_feat["weight"] / (df_feat["height"] ** 2)

In [11]:
df_feat.sample(5)

,age,weight,height,income_lpa,smoker,city,occupation,insurance_premium_category,bmi
48,36,94.8,1.66,32.690000,True,Chennai,unemployed,Medium,34.402671
91,38,119.8,1.76,28.467885,False,Bangalore,government_job,Low,38.675103
35,59,59.3,1.69,43.280000,True,Chandigarh,private_job,Medium,20.762578
4,69,62.2,1.60,3.940000,True,Indore,retired,High,24.296875
70,69,99.9,1.65,0.570000,False,Chandigarh,retired,High,36.694215


In [12]:
def age_group(age):
    if age < 25:
        return 'young'
    elif age < 45:
        return 'adult'
    elif age < 60:
        return 'middle_aged'
    return 'senior'

In [13]:
df_feat["age_group"] = df_feat["age"].apply(age_group)

In [14]:
df_feat.sample(5)

,age,weight,height,income_lpa,smoker,city,occupation,insurance_premium_category,bmi,age_group
3,22,109.4,1.55,3.340000,True,Mumbai,student,Medium,45.535900,young
17,65,90.1,1.70,2.230000,False,Delhi,retired,Medium,31.176471,senior
81,41,82.6,1.61,22.190000,True,Mysore,freelancer,High,31.866055,adult
95,36,52.8,1.57,19.640000,False,Indore,business_owner,Low,21.420747,adult
14,49,89.3,1.65,13.505166,False,Kota,government_job,Medium,32.800735,middle_aged


In [15]:
def lifestyle_risk(data):
    if data["smoker"] and data["bmi"] > 30:
        return 'high'
    elif data["smoker"] or data["bmi"] > 27:
        return 'medium'
    return 'low'

In [16]:
df_feat["lifestyle_risk"] = df_feat.apply(lifestyle_risk, axis=1)

In [17]:
df_feat.sample(5)

,age,weight,height,income_lpa,smoker,city,occupation,insurance_premium_category,bmi,age_group,lifestyle_risk
85,33,51.4,1.86,34.66,False,Chennai,private_job,Low,14.857209,adult,low
31,39,51.1,1.83,11.77,True,Lucknow,private_job,Medium,15.258742,adult,medium
38,74,111.2,1.83,1.84,True,Jaipur,retired,High,33.204933,senior,high
53,41,101.3,1.85,30.00,True,Delhi,government_job,Medium,29.598247,adult,medium
75,53,62.3,1.74,45.07,False,Hyderabad,unemployed,Low,20.577355,middle_aged,low


In [18]:
tier_1_cities = ["Mumbai", "Delhi", "Bangalore", "Chennai", "Kolkata", "Hyderabad", "Pune"]
tier_2_cities = [
    "Jaipur", "Chandigarh", "Indore", "Lucknow", "Patna", "Ranchi", "Visakhapatnam", "Coimbatore",
    "Bhopal", "Nagpur", "Vadodara", "Surat", "Rajkot", "Jodhpur", "Raipur", "Amritsar", "Varanasi",
    "Agra", "Dehradun", "Mysore", "Jabalpur", "Guwahati", "Thiruvananthapuram", "Ludhiana", "Nashik",
    "Allahabad", "Udaipur", "Aurangabad", "Hubli", "Belgaum", "Salem", "Vijayawada", "Tiruchirappalli",
    "Bhavnagar", "Gwalior", "Dhanbad", "Bareilly", "Aligarh", "Gaya", "Kozhikode", "Warangal",
    "Kolhapur", "Bilaspur", "Jalandhar", "Noida", "Guntur", "Asansol", "Siliguri"
]

In [19]:
def city_tier(city):
    if city in tier_1_cities:
        return 1
    elif city in tier_2_cities:
        return 2
    return 3

In [20]:
df_feat["city_tier"] = df_feat["city"].apply(city_tier)

In [21]:
df_feat.sample(5)

,age,weight,height,income_lpa,smoker,city,occupation,insurance_premium_category,bmi,age_group,lifestyle_risk,city_tier
23,35,70.3,1.78,23.71,False,Mysore,unemployed,Medium,22.187855,adult,low,2
58,72,95.9,1.79,3.31,True,Indore,retired,High,29.930402,senior,medium,2
6,19,80.1,1.68,3.59,True,Hyderabad,student,Medium,28.380102,young,medium,1
96,26,113.8,1.54,34.01,False,Delhi,private_job,Low,47.984483,adult,medium,1
84,75,86.2,1.73,0.62,True,Jaipur,retired,High,28.801497,senior,medium,2


In [23]:
df_feat.drop(columns=["age", "height", "weight", "city", "smoker"]).sample(5)

,income_lpa,occupation,insurance_premium_category,bmi,age_group,lifestyle_risk,city_tier
31,11.77,private_job,Medium,15.258742,adult,medium,2
38,1.84,retired,High,33.204933,senior,high,2
70,0.57,retired,High,36.694215,senior,medium,2
65,38.07,unemployed,High,37.662982,middle_aged,high,2
8,1.78,retired,Medium,23.233456,senior,low,2


Features & Target

In [24]:
X = df_feat[["age_group", "bmi", "lifestyle_risk", "city_tier", "occupation", "income_lpa"]]
y = df_feat["insurance_premium_category"]

In [25]:
X

,age_group,bmi,lifestyle_risk,city_tier,occupation,income_lpa
0,senior,49.227482,medium,2,retired,2.92000
1,adult,30.189017,medium,1,freelancer,34.28000
2,adult,21.118382,low,2,freelancer,36.64000
3,young,45.535900,high,1,student,3.34000
4,senior,24.296875,medium,2,retired,3.94000
...,...,...,...,...,...,...
95,adult,21.420747,low,2,business_owner,19.64000
96,adult,47.984483,medium,1,private_job,34.01000
97,middle_aged,18.765432,low,1,freelancer,44.86000
98,adult,30.521676,medium,1,business_owner,28.30000


In [26]:
y

0       High
1        Low
2        Low
3     Medium
4       High
       ...  
95       Low
96       Low
97       Low
98       Low
99       Low
Name: insurance_premium_category, Length: 100, dtype: object

In [27]:
X.shape

(100, 6)

In [28]:
y.shape

(100,)

Categorical & Numerical features

In [29]:
categorical_features = ["age_group", "lifestyle_risk", "occupation", "city_tier"]
numerical_features = ["bmi", "income_lpa"]

Column Transformer for OHE

In [30]:
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(), categorical_features),
        ("num", "passthrough", numerical_features)
    ]
)

Pipeline

In [31]:
pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(random_state=42))
])

Split data & train model

In [32]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [33]:
pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat', OneHotEncoder(),
                                                  ['age_group',
                                                   'lifestyle_risk',
                                                   'occupation', 'city_tier']),
                                                 ('num', 'passthrough',
                                                  ['bmi', 'income_lpa'])])),
                ('classifier', RandomForestClassifier(random_state=42))])

Predict & Evaluate

In [34]:
y_pred = pipeline.predict(X_test)

In [35]:
y_pred

array(['Medium', 'Low', 'High', 'Medium', 'High', 'Medium', 'Medium',
       'Medium', 'Low', 'High', 'Medium', 'Low', 'Medium', 'Medium',
       'Low', 'High', 'High', 'Medium', 'High', 'Medium'], dtype=object)

In [36]:
accuracy_score(y_test, y_pred)

0.45

In [37]:
X_test.sample(5)

,age_group,bmi,lifestyle_risk,city_tier,occupation,income_lpa
22,middle_aged,31.771627,medium,2,government_job,30.00
39,middle_aged,35.643424,high,1,unemployed,11.99
30,adult,29.937519,medium,1,business_owner,32.97
18,middle_aged,24.969136,medium,3,business_owner,38.14
10,adult,22.949982,medium,1,business_owner,32.78


In [38]:
import pickle

In [39]:
pickle_model_path = "model.pkl"
with open(pickle_model_path, "wb") as f:
    pickle.dump(pipeline, f)